In [1]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from time import perf_counter
from collections import OrderedDict

In [2]:
class BaselineModel:
    def __init__(self):
        self.model_name = "baseline"
    def similarity(self, first: str, second: str) -> float:
        if not first or not second:
            return 0.0

        return (sum(1 for x in first if x in second) / len(first) + \
            sum(1 for x in second if x in first) / len(second)) / 2
def show(model, w1: str, w2: str):
    print(f"{model.model_name}({w1}, {w2}) -> {model.similarity(w1, w2):.2f}")

baseline = BaselineModel()

show(baseline, "кошка", "кот")
show(baseline, "кошка", "собака")
show(baseline, "кошка", "окошко")

baseline(кошка, кот) -> 0.63
baseline(кошка, собака) -> 0.73
baseline(кошка, окошко) -> 0.90


## Метрика - Mean-AUC

Для каждой пары из топа для одного слова считаем количество пар $1<=i<j<=n$, для которых истинно $similarity(target, top_i) > similarity(target, top_j)$, после делим на количество этих пар и получаем метрику.



In [3]:
def estimate_mean_auc(model, test_df):
    mean_auc = 0.0
    for _, row in test_df.iterrows():
        tagret = row.target
        sims = [model.similarity(w, tagret) for w in row[1:]]
        good_pairs = 0
        for i in range(len(sims)):
            for j in range(i+1, len(sims)):
                if sims[i] > sims[j]:
                    good_pairs += 1
        auc = good_pairs / (len(sims)*(len(sims)-1) / 2)
        mean_auc += auc
    return mean_auc / len(test_df)


In [4]:
class TransformerModel:
    def __init__(self, model_name, maxsize=128):
        self.model_name = model_name
        self._model = SentenceTransformer(model_name)
        self._cache = OrderedDict()
        self._maxsize = maxsize

    def similarity(self, first: str, second: str) -> float:
        if not first or not second:
            return 0.0

        embeddings = self._encode(first), self._encode(second)
        return embeddings[0] @ embeddings[1]

    def _encode(self, text):
        if text in self._cache:
            self._cache.move_to_end(text)
            return self._cache[text]

        embedding = self._model.encode(text, normalize_embeddings = True)
        while len(self._cache) >= self._maxsize:
            self._cache.popitem(last=False)
        self._cache[text] = embedding

        return embedding


In [5]:
models = [BaselineModel(),
          TransformerModel('cointegrated/rubert-tiny2'),
          TransformerModel('sergeyzh/rubert-tiny-sts-v2'),
          TransformerModel('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')]

test_df = pd.read_csv('test.csv')
results = []

for model in models:
    t_start = perf_counter()
    auc = estimate_mean_auc(model, test_df)
    results.append({"model": model.model_name, "test_auc": auc, "test time": perf_counter() - t_start})

pd.DataFrame(results)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,model,test_auc,test time
0,baseline,0.477778,0.009077
1,cointegrated/rubert-tiny2,0.744444,0.139754
2,sergeyzh/rubert-tiny-sts-v2,0.800000,0.141646
3,sentence-transformers/paraphrase-multilingual-...,0.711111,0.606573


## Вот такие пироги